In [ ]:
filepaths = []
transcripts = []

In [ ]:
import csv
import numpy as np

def csv_columns_to_list(file_path):
    """Simple version for CSV that already has correct wav filenames"""
    result = []
    with open(file_path, 'r', encoding='utf-8') as file:
        reader = csv.reader(file)
        next(reader)  # skip header
        for row in reader:
            if len(row) >= 2 and len(row[0]) > 0:
                result.append([str(row[0]), str(row[1])])
    return result

filepaths_dir = r"C:\Users\sv777\Downloads\AblationStudy\SubSetAudioWAV\transcriptions.csv"

csv_data = np.array(csv_columns_to_list(filepaths_dir))  
print(csv_data)
filepaths = csv_data[:,0] 
transcripts = csv_data[:,1] 

print(filepaths)

In [ ]:
import os
from transformers import WhisperFeatureExtractor

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
model_name = "openai/whisper-large-v2"
language = "English"
language_abbr = 'en'
task = "transcribe"

feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained(model_name, language=language, task=task)
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained(model_name, language=language, task=task)

In [ ]:
import os
import re
import librosa
import soundfile as sf


all_data = []
for wav_file, transcript in zip(filepaths, transcripts):
    audio_data, sample_rate = librosa.load(r'C:\Users\sv777\Downloads\AblationStudy\SubSetAudioWAV/' + wav_file, sr=16000)
    print(transcript)
    all_data.append((feature_extractor(audio_data, sampling_rate=16000).input_features[0], tokenizer(transcript).input_ids, wav_file)) #Added wav_files to do unique participants
    # break

print(len(all_data))

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer, TrainerCallback, TrainingArguments, TrainerState, TrainerControl

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

training_args = Seq2SeqTrainingArguments(
    output_dir="temp",  
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,  
    learning_rate=1e-3,
    warmup_steps=50,
    num_train_epochs=10, 
    eval_strategy="epoch",
    fp16=True,
    per_device_eval_batch_size=8,
    generation_max_length=128,
    logging_steps=25,
    remove_unused_columns=False,  
    label_names=["labels"], 
)

In [ ]:

multi_stress_name = 'SubSetAudioWAV'

eval_dir = 'C:/Users/sv777/Downloads/AblationStudy/Script/Whisper_fine-tuning/eval/SubSetAudioWAV/'

In [ ]:
participant_folds = [
    [1001, 1022, 1033, 1044, 1065, 1071, 1003, 1010, 1025, 1055, 1082, 1091, 1061, 1013, 1005, 1039, 1080, 1081,],
    [1011, 1023, 1034, 1051, 1066, 1077, 1004, 1012, 1037, 1058, 1084, 1007, 1063, 1043, 1015, 1042, 1083, 1085,],
    [1014, 1026, 1035, 1057, 1067, 1086, 1006, 1020, 1052, 1075, 1089, 1029, 1073, 1046, 1032, 1050, 1088, 1090,],
    [1016, 1027, 1040, 1062, 1068, 1087, 1008, 1021, 1053, 1076, 1056, 1030, 1074, 1049, 1036, 1059, 1019, 1018,],
    [1017, 1028, 1041, 1064, 1069, 1002, 1009, 1024, 1054, 1078, 1072, 1060, 1048, 1079, 1038, 1070, 1045, 1031,]
]

In [ ]:
def save_list_to_file(string_list, file_path):
    with open(file_path, 'w', encoding="utf-8") as file:
        for line in string_list:
            file.write(line + '\n')

def evaluate_whisper(fold):
    test_participants = participant_folds[fold]

    test_dataset = DatasetDict()
    data_dict = {
        'input_features':[],
        'labels':[]
    }

    for i in range(len(all_data)):
        participant_id = int(all_data[i][2].split('_')[0])
        
        if participant_id in test_participants:
            data_dict['input_features'].append(all_data[i][0])
            data_dict['labels'].append(all_data[i][1])

    test_dataset["test"] = Dataset.from_dict(data_dict)
    
    model.config.use_cache = True
    
    for dataset_name in ["test"]:
        eval_dataloader = DataLoader(test_dataset[dataset_name], batch_size=8, collate_fn=data_collator)
        
        model.eval()
        results_gt = []
        results_pr = []
        for step, batch in enumerate(tqdm(eval_dataloader)):
            with torch.cuda.amp.autocast():
                with torch.no_grad():
                    generated_tokens = (
                        model.generate(
                            input_features=batch["input_features"].to("cuda"),
                            decoder_input_ids=batch["labels"][:, :4].to("cuda"),
                            max_new_tokens=255,
                            language="en"
                        )
                        .cpu()
                        .numpy()
                    )
                    labels = batch["labels"].cpu().numpy()
                    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
                    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
                    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
                    results_pr.extend(decoded_preds)
                    results_gt.extend(decoded_labels)
        save_list_to_file(results_pr, eval_dir + f'fold{fold}_pr.txt')
        save_list_to_file(results_gt, eval_dir + f'fold{fold}_gt.txt')

    return
    

In [ ]:
import random
from datasets import load_dataset, DatasetDict, Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import gc
from transformers import WhisperForConditionalGeneration, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
from peft import LoraConfig, PeftModel, LoraModel, LoraConfig, get_peft_model
from transformers.trainer_utils import PREFIX_CHECKPOINT_DIR
import torchaudio
import torch
from transformers import WhisperProcessor
import os


for fold in range(5):
    test_participants = participant_folds[fold]
    train_participants = []
    for other_fold in range(5):
        if other_fold != fold:
            train_participants.extend(participant_folds[other_fold])
    
    print(f"Test participants: {test_participants}")
    print(f"Train participants: {train_participants}")

    common_voice = DatasetDict()
    data_dict = { #For training
        'input_features':[],
        'labels':[]
    }

    for i in range(len(all_data)):
        participant_id = int(all_data[i][2].split('_')[0])
        
        if participant_id in train_participants:
            data_dict['input_features'].append(all_data[i][0])
            data_dict['labels'].append(all_data[i][1])
    
    common_voice["train"] = Dataset.from_dict(data_dict)

    data_dict = { #For testing
        'input_features':[],
        'labels':[]
    }

    for i in range(len(all_data)):
        participant_id = int(all_data[i][2].split('_')[0])
        
        if participant_id in test_participants:
            data_dict['input_features'].append(all_data[i][0])
            data_dict['labels'].append(all_data[i][1])
    
    common_voice["test"] = Dataset.from_dict(data_dict)
    
    print(f"Training samples: {len(common_voice['train'])}")
    print(f"Test samples: {len(common_voice['test'])}")
    
    print(f'TRAIN fold={fold+1}')

    # evaluate_whisper(fold)
    if True:
        model = WhisperForConditionalGeneration.from_pretrained(model_name, quantization_config=BitsAndBytesConfig(load_in_8bit=True))
        model.config.forced_decoder_ids = None
        model.config.suppress_tokens = []
        model = prepare_model_for_kbit_training(model)
        config = LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
        model = get_peft_model(model, config)
        # model.print_trainable_parameters()
        
        trainer = Seq2SeqTrainer(
            args=training_args,
            model=model,
            train_dataset=common_voice["train"],
            eval_dataset=common_voice["test"],
            data_collator=data_collator,
            tokenizer=processor.feature_extractor,
        )
        
        # model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
        trainer.train()
        model.save_pretrained(f'C:/Users/sv777/Downloads/AblationStudy/Script/Whisper_fine-tuning/models/Ablation/{multi_stress_name}{fold}', save_adapter=True, save_config=True)
    else:
        model = WhisperForConditionalGeneration.from_pretrained(model_name, quantization_config=BitsAndBytesConfig(load_in_8bit=True))
        model.config.forced_decoder_ids = None
        model.config.suppress_tokens = []
        model = prepare_model_for_kbit_training(model)
        model = PeftModel.from_pretrained(model, f'models/Ablation/{multi_stress_name}{fold}')
        #Couldn't find a dataset to use this on 
    
    evaluate_whisper(fold)